In [1]:
import pandas as pd

# Read the CSV file
df = pd.read_csv('BTC-2019min.csv')

# Convert the 'Date' column to datetime format
df['date'] = pd.to_datetime(df['date'])

# Filter the data between 4 March 2019 and 10 December 2019
start_date = pd.to_datetime('2019-03-04')
end_date = pd.to_datetime('2019-12-18')
filtered_data = df.loc[(df['date'] >= start_date) & (df['date'] <= end_date)]


filtered_data = filtered_data[['date','open','high','low','close']]

#inverse the rows,to have increasing date
filtered_data = filtered_data.iloc[::-1]

#drop the last row
filtered_data = filtered_data.drop(filtered_data.index[-1])


# Save the filtered data to a new CSV file
filtered_data.to_csv('bitcoin_minuntely.csv', index=False)

bitcoin = filtered_data

In [2]:
bitcoin.describe()

,date,open,high,low,close
count,416160,416160.000000,416160.000000,416160.000000,416160.000000
mean,2019-07-26 11:59:30.000000768,8151.063023,8156.071952,8145.874338,8150.692387
min,2019-03-04 00:00:00,3671.440000,3671.440000,3670.000000,3671.440000
25%,2019-05-15 05:59:45,7058.670000,7062.920000,7054.410000,7058.862500
50%,2019-07-26 11:59:30,8264.630000,8270.000000,8259.765000,8263.630000
75%,2019-10-06 17:59:15,10063.410000,10069.550000,10056.112500,10061.605000
max,2019-12-17 23:59:00,13853.250000,13880.000000,13830.060000,13850.570000
std,NaN,2296.973712,2299.276120,2294.439441,2296.728695


In [3]:
bitcoin.head()

,date,open,high,low,close
436319,2019-03-04 00:00:00,3789.70,3789.70,3786.85,3786.85
436318,2019-03-04 00:01:00,3789.70,3789.70,3789.41,3789.70
436317,2019-03-04 00:02:00,3790.63,3790.65,3790.63,3790.65
436316,2019-03-04 00:03:00,3790.64,3792.87,3790.64,3792.87
436315,2019-03-04 00:04:00,3793.03,3793.04,3793.03,3793.04


In [4]:
bitcoin.tail()

,date,open,high,low,close
20164,2019-12-17 23:55:00,6618.84,6619.50,6607.11,6607.11
20163,2019-12-17 23:56:00,6611.26,6614.57,6606.10,6613.69
20162,2019-12-17 23:57:00,6619.22,6619.22,6610.04,6612.57
20161,2019-12-17 23:58:00,6610.86,6618.64,6608.39,6608.39
20160,2019-12-17 23:59:00,6610.23,6618.65,6610.23,6612.30


**In the following cell, the return values are calculated over different time horizons using closed price of bitcoin**

In [1]:
import pandas as pd
import numpy as np

bitcoin = pd.read_csv('bitcoin_minuntely.csv')

bitcoin = bitcoin[['date','close']]

num_rows = bitcoin.shape[0]

end_day = pd.to_datetime('2019-12-11')

m = [1, 5, 15, 60 , 10080]
returns_mean = []
allreturns = []

for horizon in m:

  returns = list()

  i = 0

  while(pd.to_datetime(bitcoin['date'].iloc[ i ]) < end_day ):

    returns.append( (bitcoin['close'].iloc[ i + horizon ] / bitcoin['close'].iloc[i]) - 1 )
    i+=1
  
  allreturns.append(returns)

np.shape(allreturns)


(5, 406080)

**In this cell, data is split into train, val, and test sets as well as the class label for each timestep sample is calculated**


In [2]:
bitcoin = bitcoin[ bitcoin['date'] < '2019-12-11']

allreturns =  np.array(allreturns)

num_timestamps = len(bitcoin)

index1 = int(num_timestamps * 5/9)

index2 = int (num_timestamps * 6/9)

bitcoin_train = bitcoin[: index1]

bitcoin_val = bitcoin[index1 : index2]

bitcoin_test = bitcoin[index2 : ]

median_train_one = np.median(allreturns[0 , : index1])
median_train_five = np.median(allreturns[1 , : index1])
median_train_fifteen = np.median(allreturns[2 , : index1])
median_train_sixty = np.median(allreturns[3 , : index1])


bitcoin_train.loc[: , '1m return'] = allreturns[0 , : index1]
bitcoin_train.loc[: , '1m class'] = np.array(allreturns[0 , : index1] < median_train_one).astype(int)

bitcoin_train.loc[: , '5m return'] = allreturns[1 , : index1]
bitcoin_train.loc[: , '5m class'] = np.array(allreturns[1 , : index1] < median_train_five).astype(int)

bitcoin_train.loc[: , '15m return'] = allreturns[2 , : index1]
bitcoin_train.loc[: , '15m class'] = np.array(allreturns[2 , : index1] < median_train_fifteen).astype(int)

bitcoin_train.loc[: , '60m return'] = allreturns[3 , : index1]
bitcoin_train.loc[: , '60m class'] = np.array(allreturns[3 , : index1] < median_train_sixty).astype(int)

bitcoin_train.loc[: , '1week return'] = allreturns[4 , : index1]


bitcoin_val.loc[: , '1m return'] = allreturns[0 , index1: index2]
bitcoin_val.loc[: , '1m class'] = np.array(allreturns[0 , index1 :index2 ] < median_train_one).astype(int)

bitcoin_val.loc[: , '5m return'] = allreturns[1 , index1 :index2]
bitcoin_val.loc[: , '5m class'] = np.array(allreturns[1 , index1 :index2] < median_train_five).astype(int)

bitcoin_val.loc[: , '15m return'] = allreturns[2 , index1 :index2]
bitcoin_val.loc[: , '15m class'] = np.array(allreturns[2 , index1 :index2] < median_train_fifteen).astype(int)

bitcoin_val.loc[: , '60m return'] = allreturns[3 , index1 :index2]
bitcoin_val.loc[: , '60m class'] = np.array(allreturns[3 , index1 :index2] < median_train_sixty).astype(int)

bitcoin_val.loc[: , '1week return'] = allreturns[4 , index1 :index2]



bitcoin_test.loc[: , '1m return'] = allreturns[0 , index2 :]
bitcoin_test.loc[: , '1m class'] = np.array(allreturns[0 , index2 : ] < median_train_one).astype(int)

bitcoin_test.loc[: , '5m return'] = allreturns[1 , index2 :]
bitcoin_test.loc[: , '5m class'] = np.array(allreturns[1 , index2 :] < median_train_five).astype(int)

bitcoin_test.loc[: , '15m return'] = allreturns[2 , index2 :]
bitcoin_test.loc[: , '15m class'] = np.array(allreturns[2 , index2 :] < median_train_fifteen).astype(int)

bitcoin_test.loc[: , '60m return'] = allreturns[3 , index2 :]
bitcoin_test.loc[: , '60m class'] = np.array(allreturns[3 , index2 :] < median_train_sixty).astype(int)

bitcoin_test.loc[: , '1week return'] = allreturns[4 , index2 :]

bitcoin_train.to_csv('bitcoin_train_raw.csv')  
bitcoin_val.to_csv('bitcoin_val_raw.csv') 
bitcoin_test.to_csv('bitcoin_test_raw.csv')



/tmp/ipykernel_16088/3447979590.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bitcoin_train.loc[: , '1m return'] = allreturns[0 , : index1]
/tmp/ipykernel_16088/3447979590.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bitcoin_train.loc[: , '1m class'] = np.array(allreturns[0 , : index1] < median_train_one).astype(int)
/tmp/ipykernel_16088/3447979590.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value

In [3]:
bitcoin_train.describe()

,close,1m return,1m class,5m return,5m class,15m return,15m class,60m return,60m class,1week return
count,225600.000000,225600.000000,225600.000000,225600.000000,225600.000000,225600.000000,225600.000000,225600.000000,225600.000000,225600.000000
mean,7488.072500,0.000006,0.455417,0.000029,0.500000,0.000086,0.500000,0.000340,0.500000,0.056075
std,2724.039258,0.001260,0.498009,0.002729,0.500001,0.004684,0.500001,0.009155,0.500001,0.114501
min,3671.440000,-0.031645,0.000000,-0.094556,0.000000,-0.187038,0.000000,-0.197700,0.000000,-0.307595
25%,5132.655000,-0.000439,0.000000,-0.000885,0.000000,-0.001411,0.000000,-0.002437,0.000000,-0.017040
50%,7840.990000,0.000000,0.000000,0.000024,0.500000,0.000086,0.500000,0.000266,0.500000,0.035457
75%,9847.960000,0.000464,1.000000,0.000961,1.000000,0.001604,1.000000,0.003134,1.000000,0.122191
max,13850.570000,0.105122,1.000000,0.080455,1.000000,0.163627,1.000000,0.214031,1.000000,0.517274


In [6]:
bitcoin_train.head()

,date,close,1m return,1m class,5m return,5m class,15m return,15m class,60m return,60m class,1week return
0,2019-03-04 00:00:00,3786.85,0.000753,0,0.001635,0,0.004225,0,0.004634,0,0.029856
1,2019-03-04 00:01:00,3789.70,0.000251,0,0.000895,0,0.003351,0,0.004143,0,0.029055
2,2019-03-04 00:02:00,3790.65,0.000586,0,0.001195,0,0.003559,0,0.003464,0,0.028797
3,2019-03-04 00:03:00,3792.87,0.000045,0,0.000609,0,0.002043,0,0.002660,0,0.028329
4,2019-03-04 00:04:00,3793.04,0.000000,0,0.000564,0,0.001598,0,0.002681,0,0.028283


In [7]:
bitcoin_train.tail()

,date,close,1m return,1m class,5m return,5m class,15m return,15m class,60m return,60m class,1week return
225595,2019-08-07 15:55:00,11609.22,0.001181,0,0.003852,0,0.005206,0,0.001532,0,-0.099285
225596,2019-08-07 15:56:00,11622.93,0.002552,0,0.004852,0,0.005115,0,0.001422,0,-0.101173
225597,2019-08-07 15:57:00,11652.59,0.000242,0,0.003381,0,0.002196,0,-0.003187,1,-0.104403
225598,2019-08-07 15:58:00,11655.41,-0.000177,1,0.001703,0,0.000516,0,-0.002089,1,-0.103465
225599,2019-08-07 15:59:00,11653.35,0.000051,0,0.001532,0,-0.000072,1,-0.000857,1,-0.101401


Loading and splitting the twitter features for training the model:


In [2]:
import pandas as pd

bitcoin_train = pd.read_csv('bitcoin_train_raw.csv')  
bitcoin_val = pd.read_csv('bitcoin_val_raw.csv') 
bitcoin_test = pd.read_csv('bitcoin_test_raw.csv')

tweets_features_all = pd.read_csv('The features of twitter.csv')
tweets_features_all.drop(tweets_features_all.columns[0],axis=1, inplace=True)


index1 = bitcoin_train.shape[0]
index2 = index1 + bitcoin_val.shape[0]

sentiment_train = tweets_features_all[:index1]
sentiment_val = tweets_features_all[index1 : index2]
sentiment_test = tweets_features_all[index2 :]


sentiment_train.shape[0] == bitcoin_train.shape[0] , sentiment_val.shape[0] == bitcoin_val.shape[0] , sentiment_test.shape[0]== bitcoin_test.shape[0]

(True, True, True)

In [3]:
sentiment_train.head()

,date,num_tweets,sum_score,Weighted_Sentiment_Score
0,2019-03-04 00:00:00,0,0.0,0.00
1,2019-03-04 00:01:00,0,0.0,0.00
2,2019-03-04 00:02:00,0,0.0,0.00
3,2019-03-04 00:03:00+00:00,1,0.2,0.26
4,2019-03-04 00:04:00+00:00,0,0.0,0.00


In [4]:
sentiment_train.describe()

,num_tweets,sum_score,Weighted_Sentiment_Score
count,225600.000000,225600.000000,225600.000000
mean,0.284929,-0.004789,-0.000411
std,0.620359,0.165955,0.221101
min,0.000000,-2.000000,-4.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000
max,17.000000,2.400000,6.570000


In [5]:
import warnings
warnings.filterwarnings('ignore')

Following we standardize the bitcoin return features.

In [6]:
from sklearn.preprocessing import StandardScaler

def standardization( dataframe1, dataframe2, dataframe3 ,startcolumns , step ):

  for i in range (startcolumns , len(dataframe1.columns) ,step):
    
    retrn_data = dataframe1.iloc[:, i].values.reshape(-1, 1)

    scaler = StandardScaler()
    retrn_data_standardized = scaler.fit_transform(retrn_data)

    dataframe1[dataframe1.columns[i]] = retrn_data_standardized.flatten()


##########validation data
    retrn_data = dataframe2.iloc[:, i].values.reshape(-1, 1)

    retrn_data_standardized = scaler.transform(retrn_data)

    dataframe2[dataframe2.columns[i]] = retrn_data_standardized.flatten()

##########test data

    retrn_data = dataframe3.iloc[:, i].values.reshape(-1, 1)

    retrn_data_standardized = scaler.transform(retrn_data)

    dataframe3[dataframe3.columns[i]] = retrn_data_standardized.flatten()




  return dataframe1 , dataframe2, dataframe3

bitcoin_train , bitcoin_val , bitcoin_test = standardization(bitcoin_train , bitcoin_val , bitcoin_test ,3 ,2 )



In [7]:
bitcoin_train.describe()

,Unnamed: 0,close,1m return,1m class,5m return,5m class,15m return,15m class,60m return,60m class,1week return
count,225600.000000,225600.000000,2.256000e+05,225600.000000,2.256000e+05,225600.000000,2.256000e+05,225600.000000,2.256000e+05,225600.000000,2.256000e+05
mean,112799.500000,7488.072500,-6.015677e-18,0.455417,3.023586e-18,0.500000,-1.385810e-18,0.500000,3.275552e-18,0.500000,9.272331e-17
std,65125.254702,2724.039258,1.000002e+00,0.498009,1.000002e+00,0.500001,1.000002e+00,0.500001,1.000002e+00,0.500001,1.000002e+00
min,0.000000,3671.440000,-2.511992e+01,0.000000,-3.465509e+01,0.000000,-3.995349e+01,0.000000,-2.163137e+01,0.000000,-3.176132e+00
25%,56399.750000,5132.655000,-3.532706e-01,0.000000,-3.346314e-01,0.000000,-3.196490e-01,0.000000,-3.033058e-01,0.000000,-6.385566e-01
50%,112799.500000,7840.990000,-4.583671e-03,0.000000,-1.613230e-03,0.500000,1.188938e-04,0.500000,-8.091715e-03,0.500000,-1.800704e-01
75%,169199.250000,9847.960000,3.637816e-01,1.000000,3.415563e-01,1.000000,3.242336e-01,1.000000,3.051950e-01,1.000000,5.774237e-01
max,225599.000000,13850.570000,8.342652e+01,1.000000,2.946737e+01,1.000000,3.491826e+01,1.000000,2.334102e+01,1.000000,4.027902e+00


In [8]:
bitcoin_val.describe()

,Unnamed: 0,close,1m return,1m class,5m return,5m class,15m return,15m class,60m return,60m class,1week return
count,45120.00000,45120.000000,45120.000000,45120.000000,45120.000000,45120.000000,45120.000000,45120.000000,45120.000000,45120.000000,45120.000000
mean,248159.50000,10465.962560,-0.006039,0.474158,-0.014020,0.516578,-0.024526,0.526574,-0.049268,0.536215,-0.677111
std,13025.16641,645.280724,0.797989,0.499337,0.748707,0.499731,0.733278,0.499299,0.743857,0.498692,0.582971
min,225600.00000,9333.690000,-15.279462,0.000000,-16.399836,0.000000,-11.796094,0.000000,-7.223189,0.000000,-2.200363
25%,236879.75000,10082.020000,-0.400915,0.000000,-0.358659,0.000000,-0.342429,0.000000,-0.332343,0.000000,-0.986755
50%,248159.50000,10362.545000,-0.004584,0.000000,-0.013571,1.000000,-0.029070,1.000000,-0.044976,1.000000,-0.775982
75%,259439.25000,10718.792500,0.378756,1.000000,0.326081,1.000000,0.293334,1.000000,0.250344,1.000000,-0.407521
max,270719.00000,12050.880000,11.744874,1.000000,13.927415,1.000000,10.409119,1.000000,6.270022,1.000000,0.767698


In [9]:
bitcoin_test.describe()

,Unnamed: 0,close,1m return,1m class,5m return,5m class,15m return,15m class,60m return,60m class,1week return
count,135360.000000,135360.000000,135360.000000,135360.000000,135360.000000,135360.000000,135360.000000,135360.000000,135360.000000,135360.000000,135360.000000
mean,338399.500000,8561.705914,-0.006349,0.468262,-0.014788,0.518898,-0.025902,0.529536,-0.052617,0.547909,-0.696023
std,39075.210556,969.418938,0.817408,0.498994,0.757411,0.499645,0.735280,0.499129,0.750520,0.497701,0.730777
min,270720.000000,6525.420000,-16.750283,0.000000,-18.556985,0.000000,-18.026280,0.000000,-15.357181,0.000000,-2.493569
25%,304559.750000,7927.230000,-0.371003,0.000000,-0.311555,0.000000,-0.296262,0.000000,-0.301935,0.000000,-1.023366
50%,338399.500000,8334.445000,-0.004584,0.000000,-0.011841,1.000000,-0.026361,1.000000,-0.051370,1.000000,-0.683449
75%,372239.250000,9268.817500,0.346306,1.000000,0.279998,1.000000,0.244329,1.000000,0.198352,1.000000,-0.393988
max,406079.000000,10593.400000,20.118534,1.000000,26.006182,1.000000,20.695029,1.000000,12.412236,1.000000,2.121062


Following we standardize the tweets' features.

In [10]:
sentiment_train, sentiment_val , sentiment_test = standardization(sentiment_train, sentiment_val ,sentiment_test, 1, 1)

In [11]:
sentiment_train.describe()

,num_tweets,sum_score,Weighted_Sentiment_Score
count,2.256000e+05,2.256000e+05,2.256000e+05
mean,1.259828e-17,2.217296e-17,-1.562186e-17
std,1.000002e+00,1.000002e+00,1.000002e+00
min,-4.592980e-01,-1.202260e+01,-1.808948e+01
25%,-4.592980e-01,2.885726e-02,1.857049e-03
50%,-4.592980e-01,2.885726e-02,1.857049e-03
75%,-4.592980e-01,2.885726e-02,1.857049e-03
max,2.694424e+01,1.449061e+01,2.971687e+01


In [12]:
sentiment_test.describe()

,num_tweets,sum_score,Weighted_Sentiment_Score
count,135360.000000,135360.000000,135360.000000
mean,-0.127972,-0.006809,-0.007871
std,0.785428,0.829729,0.833127
min,-0.459298,-12.022600,-16.823082
25%,-0.459298,0.028857,0.001857
50%,-0.459298,0.028857,0.001857
75%,-0.459298,0.028857,0.001857
max,15.660432,17.503470,30.621436


In [13]:
sentiment_val.describe()

,num_tweets,sum_score,Weighted_Sentiment_Score
count,45120.000000,45120.000000,45120.000000
mean,-0.138297,0.004738,-0.010902
std,0.768307,0.813455,0.793814
min,-0.459298,-8.407163,-17.411050
25%,-0.459298,0.028857,0.001857
50%,-0.459298,0.028857,0.001857
75%,-0.459298,0.028857,0.001857
max,14.048459,8.464877,18.907298


In [14]:
import numpy as np
def data_generator_for_model(df , columnnames):

  window_size = 120
  slide_step = 1

  classes = df[columnnames[-1]]

  df = df[columnnames[:-1]]
  
  X = np.array([df[i:i+window_size] for i in range(0, len(df)-window_size+1, slide_step)])[:-1 , : , :]

  Y = np.array([classes[i] for i in range(120,len(classes))])
  
  return X , Y


 

Following the data is ready to feed to the model for training, validating, and testing :

In [15]:
dataset_train = pd.concat([sentiment_train ,  bitcoin_train.drop(bitcoin_train.columns[0:2], axis = 1)], axis = 1 )

columns_name = ['num_tweets', 'sum_score', 'Weighted_Sentiment_Score' , '60m return' , '1week return', '60m class']

X_train , Y_train = data_generator_for_model(dataset_train , columns_name)

X_train.shape , Y_train.shape

((225480, 120, 5), (225480,))

In [16]:
dataset_train.head()

,date,num_tweets,sum_score,Weighted_Sentiment_Score,close,1m return,1m class,5m return,5m class,15m return,15m class,60m return,60m class,1week return
0,2019-03-04 00:00:00,-0.459298,0.028857,0.001857,3786.85,0.592730,0,0.588403,0,0.883810,0,0.469118,0,-0.228989
1,2019-03-04 00:01:00,-0.459298,0.028857,0.001857,3789.70,0.194371,0,0.317246,0,0.697208,0,0.415416,0,-0.235984
2,2019-03-04 00:02:00,-0.459298,0.028857,0.001857,3790.65,0.460226,0,0.427352,0,0.741526,0,0.341248,0,-0.238236
3,2019-03-04 00:03:00+00:00,1.152675,1.234003,1.177794,3792.87,0.030989,0,0.212644,0,0.417958,0,0.253480,0,-0.242321
4,2019-03-04 00:04:00+00:00,-0.459298,0.028857,0.001857,3793.04,-0.004584,0,0.196213,0,0.322807,0,0.255771,0,-0.242723


In [17]:
bitcoin_val.reset_index(drop=True, inplace=True)
sentiment_val.reset_index(drop=True, inplace=True)


dataset_val = pd.concat([sentiment_val , bitcoin_val.drop(bitcoin_val.columns[0:2]  , axis = 1)] , axis = 1 )

columns_name = ['num_tweets', 'sum_score', 'Weighted_Sentiment_Score' , '60m return' , '1week return', '60m class']

X_val , Y_val = data_generator_for_model(dataset_val , columns_name)

X_val.shape, Y_val.shape

((45000, 120, 5), (45000,))

In [18]:
dataset_val.head()

,date,num_tweets,sum_score,Weighted_Sentiment_Score,close,1m return,1m class,5m return,5m class,15m return,15m class,60m return,60m class,1week return
0,2019-08-07 16:00:00+00:00,1.152675,0.028857,0.001857,11653.94,1.723859,0,0.786169,0,0.109383,0,-0.155938,1,-1.350919
1,2019-08-07 16:01:00+00:00,-0.459298,0.028857,0.001857,11679.32,0.856400,0,-0.026815,1,-0.264930,1,-0.325609,1,-1.334189
2,2019-08-07 16:02:00+00:00,2.764648,-1.778861,-0.812253,11691.99,-1.140231,1,-0.622827,1,-0.165320,1,-0.545863,1,-1.344822
3,2019-08-07 16:03:00+00:00,-0.459298,0.028857,0.001857,11675.26,-0.280575,1,0.190342,0,-0.125481,1,-0.465387,1,-1.323996
4,2019-08-07 16:04:00+00:00,-0.459298,0.028857,0.001857,11671.20,0.544871,0,0.606994,0,-0.076856,1,-0.379903,1,-1.332016


In [19]:
bitcoin_test.reset_index(drop=True, inplace=True)
sentiment_test.reset_index(drop=True, inplace=True)


dataset_test = pd.concat([sentiment_test , bitcoin_test.drop(bitcoin_test.columns[0:2]  , axis = 1)] , axis = 1 )

columns_name = ['num_tweets', 'sum_score', 'Weighted_Sentiment_Score' , '60m return' , '1week return', '60m class']

X_test , Y_test = data_generator_for_model(dataset_test , columns_name)

X_test.shape, Y_test.shape

((135240, 120, 5), (135240,))

In [20]:
dataset_test.head()

,date,num_tweets,sum_score,Weighted_Sentiment_Score,close,1m return,1m class,5m return,5m class,15m return,15m class,60m return,60m class,1week return
0,2019-09-08 00:00:00+00:00,1.152675,-4.791726,-5.787369,10486.56,1.794420,0,0.492272,0,0.604517,0,0.228410,0,-0.599022
1,2019-09-08 00:01:00+00:00,-0.459298,0.028857,0.001857,10510.33,0.431124,0,-0.337490,1,0.333331,0,0.185512,0,-0.618493
2,2019-09-08 00:02:00+00:00,-0.459298,0.028857,0.001857,10516.10,-0.360053,1,-0.261706,1,0.220860,0,0.084224,0,-0.617326
3,2019-09-08 00:03:00+00:00,-0.459298,0.028857,0.001857,10511.39,-0.658457,1,-0.041176,1,0.021904,0,0.073263,0,-0.612149
4,2019-09-08 00:04:00+00:00,-0.459298,0.028857,0.001857,10502.73,-0.139093,1,0.190089,0,0.288657,0,0.109650,0,-0.611751


following we save the variables of X and Y for train, val, and test as a pickle object so later they can be loaded for training the model

In [27]:
import pickle 

filepath = ''

with open(filepath+'X_train.pickle', 'wb') as file:
    # Serialize the object and write to file
    pickle.dump(X_train, file)
  
with open(filepath+'Y_train.pickle', 'wb') as file:
    # Serialize the object and write to file
    pickle.dump(Y_train, file)
  
with open(filepath+'X_val.pickle', 'wb') as file:
    # Serialize the object and write to file
    pickle.dump(X_val, file)
  
with open(filepath+'Y_val.pickle', 'wb') as file:
    # Serialize the object and write to file
    pickle.dump(Y_val, file)
  
with open(filepath+'X_test.pickle', 'wb') as file:
    # Serialize the object and write to file
    pickle.dump(X_test, file)
  
with open(filepath+'Y_test.pickle', 'wb') as file:
    # Serialize the object and write to file
    pickle.dump(Y_test, file)
  


So far, the data regarding 60 minute retursn were processed and saved to file. In the following the data ragarding other time horizons will be generated for models.

In [ ]:
#write 1m instead of 60min
import pickle

columns_name_5 = ['num_tweets', 'sum_score', 'Weighted_Sentiment_Score' , '5m return' , '1week return', '5m class']
columns_name_15 = ['num_tweets', 'sum_score', 'Weighted_Sentiment_Score' , '15m return' , '1week return', '15m class']
columns_name_60 = ['num_tweets', 'sum_score', 'Weighted_Sentiment_Score' , '60m return' , '1week return', '60m class']

def save_pickle (x, filename):

  with open(filename, 'wb') as file:
    pickle.dump(x, file)


data_dict = {'train' : dataset_train , 'val' : dataset_val , 'test' : dataset_test}


def genet_save(namecolumns , stringreturn , filepath):

  for key in data_dict:
    
    X , Y = data_generator_for_model(data_dict[key], namecolumns)
    print(Y)
    save_pickle(X , filepath+'X' + '_' + key + '_' + stringreturn + '.pickle')
    save_pickle(Y , filepath+'Y' + '_' + key + '_' + stringreturn + '.pickle')

genet_save(columns_name_5 , '5min', '/content/drive/MyDrive/Dataset_with_memory /5min/')
genet_save(columns_name_15 , '15min', '/content/drive/MyDrive/Dataset_with_memory /15min/')
genet_save(columns_name_60 , '60min', '/content/drive/MyDrive/Dataset_with_memory /60min/')











  

[1 1 0 ... 0 0 0]
[0 0 1 ... 0 0 0]
[0 0 0 ... 1 0 0]
[1 1 1 ... 0 0 0]
[1 1 1 ... 0 0 0]
[0 0 1 ... 0 0 0]
[1 1 1 ... 1 1 1]
[1 1 1 ... 0 0 0]
[1 1 1 ... 0 0 0]
